In [1]:

import dataclasses as dc
from pathlib import Path
import re

iso_slug_regex = re.compile(r'(\d+)([A-Z][a-zA-Z]*)(\d*)')
atom_num_regex = re.compile(r'\((\d+)([A-Z][a-zA-Z]*)\)(\d*)')

def iso_slug_to_iso_name(iso_slug):
	iso_name = ''
	for match in iso_slug_regex.finditer(iso_slug):
		iso_mass = int(match[1])
		iso_atom = match[2]
		iso_multiplicity = match[3]
		iso_name += f'({iso_mass}{iso_atom}){iso_multiplicity}'
	return iso_name

def iso_name_to_iso_spec(iso_name):
	iso_spec = []
	for match in atom_num_regex.finditer(iso_name):
		iso_spec.append(
			((int(match[1]), match[2]), int(match[3]) if len(match[3]) > 0 else 1)
		)
	return tuple(iso_spec)

def mol_spec_from_iso_name(iso_name):
	mol_spec : dict[str,int] = dict()
	for match in atom_num_regex.finditer(iso_name):
		atom_name = match[2]
		atom_multiplicity = int(match[3]) if len(match[3]) > 0 else 1
		mol_spec[atom_name] = mol_spec.get(atom_name, 0) + atom_multiplicity
	
	for k in tuple(mol_spec.keys()):
		if mol_spec[k] == 0:
			del mol_spec
	
	return tuple(sorted((k,mol_spec[k]) for k in sorted(mol_spec.keys())))

def parse_pc_data_file_name(fname : str):
	iso_slug, remainder = fname.split('__', 1)
	remainder, ftype = remainder.rsplit('.', 1)
	dsname, t_cont = remainder.rsplit('_T', 1)
	
	iso_name = iso_slug_to_iso_name(iso_slug)
	
	return (
		ftype,
		iso_name,
		mol_spec_from_iso_name(iso_name),
		dsname,
		t_cont
	)

@dc.dataclass
class PCDataFileSet:
	t_cont : float
	mol_spec : tuple
	iso_name : str
	s_max : float = 1E-26
	contbins : None | Path = None
	continuum : None | Path = None
	stronglines : None | Path = None
	

pc_data_files = dict()

for pc_data_file in Path('./test_data/EXOMOL_test').iterdir():
	ftype, iso_name, mol_spec, dsname, t_cont = parse_pc_data_file_name(pc_data_file.name)
	
	setattr(
		pc_data_files
			.setdefault(mol_spec, dict())
				.setdefault(iso_name,dict())
					.setdefault(dsname,dict())
						.setdefault(
							float(t_cont),
							PCDataFileSet(
								t_cont=float(t_cont),
								mol_spec = mol_spec,
								iso_name = iso_name,
							)
						),
		ftype,
		pc_data_file
	)


for mol_spec,a in pc_data_files.items():
	print(f'{mol_spec}')
	for iso_name, b in a.items():
		print(f'\t{iso_name}')
		for dsname, c in b.items():
			print(f'\t\t{dsname}')
			for t_cont, d in c.items():
				print(f'\t\t\t{t_cont}\n\t\t\t\t{d}')







(('C', 1), ('H', 4))
	(12C)(1H)4
		YT34to10
			250.0
				PCDataFileSet(t_cont=250.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.stronglines'))
			200.0
				PCDataFileSet(t_cont=200.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.stronglines'))
			300.0
				PCDataFileSet(t_cont=300.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T300.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T3

In [2]:
from archnemesis.database.filetypes.ans_pseudo_continuum_file import AnsPseudoContinuumFile
from archnemesis.database.filetypes.ans_line_data_file import AnsLineDataFile


ans_pc_file = AnsPseudoContinuumFile(Path('./test_data/test_pc.h5'))
ans_ld_file = AnsLineDataFile(Path('./test_data/test_ld.h5'))

In [3]:
from typing import Self

import numpy as np

class DtypeStringParser:
	def __init__(self):
		self.TOK_SEP = ';'
		self.TOK_EOF = 'END_OF_FILE'
		
		self.reset()
		
	
	def reset(self, s : str | None = None) -> Self:
		self.pos = 0
		self.end = len(s) if s is not None else None
		self.current_token = None
		self.exhausted = False
		return self
		
	def next_token(self, s) -> str:
		idx = s.find(self.TOK_SEP, self.pos, self.end)
		
		if idx > 0:
			self.current_token = s[self.pos:idx]
			self.pos = idx + len(self.TOK_SEP)
		else:
			if not self.exhausted:
				self.pos = len(s)
				self.current_token = '}'
				self.exhausted = True
			else:
				self.current_token = self.TOK_EOF
		
		return self.current_token
	
	def is_exhausted(self, s) -> bool:
		return self.exhausted
	
	def parse_structured(self, s : str) -> np.dtype:
		"""
		Parse a string for a structured dtype
		"""
		fnames = []
		foffsets = []
		fdtypes = []
		
		while (tok := self.next_token(s)) != '}':
			#print(f'110 :: {tok}')
			fnames.append(tok)
			
			tok = self.next_token(s)
			#print(f'120 :: {tok}')
			foffsets.append(int(tok))
			
			fdtype = self.parse(s)
			fdtypes.append(fdtype)
		
		result = np.dtype({'names':fnames, 'formats' : fdtypes, 'offsets':foffsets})
		
		#print(f'130 :: {self.current_token}')
		assert self.current_token == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse_array(self, s : str) -> np.dtype:
		"""
		Parse a string for an array dtype
		"""
		
		tok = self.next_token(s)
		#print(f'210 :: {tok}')
		shape = tuple(int(x.strip()) for x in tok[1:-1].split(','))
		
		fdtype = self.parse(s)
		
		result = np.dtype((fdtype,shape))
		
		tok = self.next_token(s)
		#print(f'220 :: {tok}')
		assert tok == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse_type(self, s : str) -> np.dtype:
		"""
		Parse a string for a simple dtype
		"""
		tok = self.next_token(s)
		#print(f'310 :: {tok}')
		result = np.dtype(tok)
		
		tok = self.next_token(s)
		#print(f'320 :: {tok}')
		assert tok == '}', f'Expected token "}}", but got token "{tok}"'
		return result
	
	def parse(self, s : str) -> np.dtype:
		"""
		Parse a string that specifies a general dtype
		"""
		tok = self.next_token(s)
		
		result = None
		
		if tok == 'STRUCTURED{':
			#print(f'100 :: {tok}')
			result = self.parse_structured(s)
		elif tok == 'ARRAY{':
			#print(f'200 :: {tok}')
			result = self.parse_array(s)
		elif tok == 'TYPE{':
			#print(f'300 :: {tok}')
			result = self.parse_type(s)
		elif tok == '}':
			raise RuntimeError('Unexpected end of section')
		elif tok == self.TOK_EOF:
			raise RuntimeError('Unexpected end of input')
		else:
			raise RuntimeError(f'Unknown Token "{tok}"')
		
		return result


_dtype_string_parser = DtypeStringParser()

def dtype_from_string(string : str) -> np.dtype:
	return _dtype_string_parser.reset().parse(string)

In [4]:


def read_cont_data(pc_dfs):
	cont_bin_edge = np.fromfile(pc_dfs.contbins)
	#print(f'{cont_bin_edge=}')
	cont_bin_center = 0.5*(cont_bin_edge[:-1] +cont_bin_edge[1:])
	#print(f'{cont_bin_center=}')
	cont_bin_width = np.diff(cont_bin_edge)
	#print(f'{cont_bin_width=}')


	with open(pc_dfs.continuum, 'rb') as f:
		hdr = b''
		while len(hdr) < 1024 and (len(hdr) == 0 or hdr[-1] != 0):
			hdr += f.read(4)
		
		print(f'{hdr=}')

		continuum_dtype = dtype_from_string(hdr.decode('ascii'))
		print(f'{continuum_dtype=}')
		
		cont_data = np.fromfile(f, dtype=continuum_dtype)
		print(f'{cont_data[:5]=}')

	with open(pc_dfs.stronglines, 'rb') as f:
		hdr = b''
		while len(hdr) < 1024 and (len(hdr) == 0 or hdr[-1] != 0):
			hdr += f.read(4)
		
		print(f'{hdr=}')

		stronglines_dtype = dtype_from_string(hdr.decode('ascii'))
		print(f'{stronglines_dtype=}')
		
		stronglines_data = np.fromfile(f, dtype=stronglines_dtype)
		print(f'{stronglines_data[:5]=}')
	
	return cont_bin_center, cont_bin_width, cont_data, stronglines_data

In [5]:


from archnemesis.Data.gas_data import gas_info, atom_mass

atom_names = tuple(atom_mass.keys())

def mol_name_to_mol_spec(mol_name, atom_names = atom_names):
	mol_spec = dict()
	
	while len(mol_name) > 0:
		atom_name_found = False
		for atom_name in atom_names:
			
			if mol_name.startswith(atom_name):
				#print(f'{atom_name=}')
				atom_name_found = True
				atom_multiplicity = 1
				mol_name = mol_name[len(atom_name):]
				#print(f'{mol_name=}')
				i=0
				while i<len(mol_name) and mol_name[i].isdigit():
					i+=1
				if i > 0:
					atom_multiplicity = int(mol_name[:i])
					mol_name = mol_name[i:]
				mol_spec[atom_name] = mol_spec.get(atom_name, 0) + atom_multiplicity
	
	return tuple(sorted((k,v) for k,v in mol_spec.items()))
	
	
def get_rt_mol_iso_ids(pc_dfs, gas_info=gas_info):
	rt_mol_id = None
	rt_iso_id = None

	uncontained_iso_atom_regex = re.compile(r'[A-Z][a-zA-Z]*(?!\))')
	uncontained_iso_atom_subs = {'H' : '(1H)', 'D' : '(2H)'}

	for mol_id, mol_data in gas_info.items():
		mol_name = mol_data['name']
		
		rt_mol_spec = mol_name_to_mol_spec(mol_name, atom_names=atom_names)
		if rt_mol_spec != pc_dfs.mol_spec:
			continue
		
		print(f'Found molecule name {mol_name} {mol_id}')
		rt_mol_id = int(mol_id)
		
		for iso_id, iso_data in mol_data['isotope'].items():
			iso_name = iso_data['name']
			canonical_iso_name = ''
			i = 0
			j = 0
			for match in uncontained_iso_atom_regex.finditer(iso_name):
				j = match.start()
				canonical_iso_name += iso_name[i:j]
				canonical_iso_name += uncontained_iso_atom_subs[match[0]]
				i = match.end()
			canonical_iso_name += iso_name[i:]
			#print(f'{pc_dfs.iso_name=}')
			#print(f'{canonical_iso_name=}')
			if canonical_iso_name != pc_dfs.iso_name:
				continue
			rt_iso_id = int(iso_id)
			break
		
		if rt_iso_id is None:
			raise RuntimeError(f'Could not fine RADTRAN local iso id for {pc_dfs.iso_name}')
		
		break

	if rt_mol_id is None:
		raise RuntimeError(f'Could not fine RADTRAN molecule id for {pc_dfs.mol_spec}')
	
	print(f'{rt_mol_id=} {rt_iso_id=}')
	return rt_mol_id, rt_iso_id




#ans_pc_file.dump()


In [ ]:
from archnemesis.database.data_holders.pseudo_continuum_data_holder import PseudoContinuumDataHolder
from archnemesis.database.data_holders.pseudo_continuum_broadener_part import PseudoContinuumBroadenerPart
from archnemesis.database.data_holders.line_data_holder import LineDataHolder
from archnemesis.database.data_holders.line_broadener_holder import LineBroadenerHolder

for a in pc_data_files.values():
	for b in a.values():
		for c in b.values():
			for pc_dfs in c.values():
				
				print(f'{pc_dfs=}')
				cont_bin_center, cont_bin_width, cont_data, stronglines_data = read_cont_data(pc_dfs)
				
				rt_mol_id, rt_iso_id = get_rt_mol_iso_ids(pc_dfs)
				print(f'LOOP: {rt_mol_id=} {rt_iso_id=}')
				
				broadener_names = tuple(x[len("strength_weighted_gamma_"):] for x in cont_data.dtype.names if (x.startswith("strength_weighted_gamma_") and not x.endswith('self')))

				pc_dh = PseudoContinuumDataHolder(
					"EXOMOL_test",
					"This data is a test that has been extracted from exomol",
					
					t_cont = pc_dfs.t_cont,
					s_max = pc_dfs.s_max, 
					
					mol_id = rt_mol_id * np.ones_like(cont_bin_center, dtype=int),
					local_iso_id = rt_iso_id * np.ones_like(cont_bin_center, dtype=int),
					
					wn_bin_center = cont_bin_center,
					wn_bin_width = cont_bin_width,
					line_strength_sum = cont_data['line_strength_sum'],
					line_strength_weighted_mean_lower_energy_state = cont_data['strength_weighted_sum_E"'] / cont_data['line_strength_sum'],
					
					# self broadening
					line_strength_weighted_gamma_self = cont_data['strength_weighted_gamma_self'] / cont_data['line_strength_sum'],
					line_strength_weighted_n_self = cont_data['strength_weighted_n_self'] / cont_data['line_strength_sum'],
					
					# foreign broadening
					broadeners = tuple(
						PseudoContinuumBroadenerPart(
							name = x,
							line_strength_weighted_gamma_amb = cont_data[f'strength_weighted_gamma_{x}'] / cont_data['line_strength_sum'],
							line_strength_weighted_n_amb = cont_data[f'strength_weighted_n_{x}'] / cont_data['line_strength_sum'],
						) for x in broadener_names
					),
					
				)


				ans_pc_file.add_source_data(
					pc_dh.name,
					pc_dh,
					pc_dh.description
				)
				
				print('ADDED PSEUDO CONTINUUM DATA...')
				
				
				ld_dh = LineDataHolder(
					"EXOMOL_test",
					"This data is a test that has been extracted from exomol",
					
					s_min = pc_dfs.s_max,
					t_ref = pc_dfs.t_cont,
					
					mol_id = np.ones_like(stronglines_data['wavenumber'], dtype=int)*rt_mol_id,
					local_iso_id = np.ones_like(stronglines_data['wavenumber'], dtype=int)*rt_iso_id,
					nu = stronglines_data['wavenumber'],
					sw = stronglines_data['spec_line_intensity'],
					a = stronglines_data['einstein_A'],
					elower = stronglines_data['E"'],
					
					gamma_self = stronglines_data['gamma_self'],
					n_self = stronglines_data['n_self'],
					
					broadeners = tuple(
						LineBroadenerHolder(
							name = x.upper() if x in ('air',) else x,
							gamma_amb = stronglines_data[f'gamma_{x}'],
							n_amb = stronglines_data[f'n_{x}'],
							delta_amb = np.zeros_like(stronglines_data[f'gamma_{x}']),
						) for x in broadener_names
					),
				)
				
				ans_ld_file.add_source_data(
					ld_dh.name,
					ld_dh,
					ld_dh.description
				)
				
				print('ADDED LINE DATA...')
				
				

/tmp/ipykernel_1957143/1564168319.py:32: RuntimeWarning: invalid value encountered in divide
  line_strength_weighted_mean_lower_energy_state = cont_data['strength_weighted_sum_E"'] / cont_data['line_strength_sum'],
/tmp/ipykernel_1957143/1564168319.py:35: RuntimeWarning: invalid value encountered in divide
  line_strength_weighted_gamma_self = cont_data['strength_weighted_gamma_self'] / cont_data['line_strength_sum'],
/tmp/ipykernel_1957143/1564168319.py:36: RuntimeWarning: invalid value encountered in divide
  line_strength_weighted_n_self = cont_data['strength_weighted_n_self'] / cont_data['line_strength_sum'],
/tmp/ipykernel_1957143/1564168319.py:42: RuntimeWarning: invalid value encountered in divide
  line_strength_weighted_gamma_amb = cont_data[f'strength_weighted_gamma_{x}'] / cont_data['line_strength_sum'],
/tmp/ipykernel_1957143/1564168319.py:43: RuntimeWarning: invalid value encountered in divide
  line_strength_weighted_n_amb = cont_data[f'strength_weighted_n_{x}'] / cont_d

pc_dfs=PCDataFileSet(t_cont=250.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T250.0.stronglines'))
hdr=b'STRUCTURED{;line_strength_sum;0;TYPE{;<f8;};strength_weighted_sum_E";8;TYPE{;<f8;};strength_weighted_gamma_self;16;TYPE{;<f8;};strength_weighted_n_self;24;TYPE{;<f8;};strength_weighted_gamma_air;32;TYPE{;<f8;};strength_weighted_n_air;40;TYPE{;<f8;};strength_weighted_gamma_H2;48;TYPE{;<f8;};strength_weighted_n_H2;56;TYPE{;<f8;};strength_weighted_gamma_He;64;TYPE{;<f8;};strength_weighted_n_He;72;TYPE{;<f8;};strength_weighted_gamma_CO2;80;TYPE{;<f8;};strength_weighted_n_CO2;88;TYPE{;<f8;};}\x00\x00\x00\x00'
continuum_dtype=dtype([('line_strength_sum', '<f8'), ('strength_weighted_sum_E"', '<f8'), ('strength_weighted_gamma_self', '<f8'), ('stre

INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/sources/EXOMOL_test/line_data" in "test_data/test_ld.h5" succeeded
INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/line_data" in "test_data/test_ld.h5" succeeded
DEBUG :: _update_virtual_datasets :: ans_pseudo_continuum_file.py-78 :: s_name='/sources/EXOMOL_test' s_file='.' s_path='/sources/EXOMOL_test/pseudo_continuum'


mol_grp_name='CH4' iso_grp_name='1'
leaf_grp_name='line_set_0000'
leaf_grp_name='line_set_0001'
Performing update...
mol_grp_name='CH4' iso_grp_name='1'
Deleted existing leaf groups...
iso_grp_src_list has been sorted...
len(sorted_iso_grp_src_list)=2
len(sorted_iso_grp_src_list[0])=2
len(sorted_iso_grp_src_list[0][0])=2
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(200.0))
Write virtual dataset
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(250.0))
Write virtual dataset
ADDED LINE DATA...
pc_dfs=PCDataFileSet(t_cont=200.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T200.0.stronglines'))
hdr=b'STRUCTURED{;line_strength_sum;0;TYPE{;<f8;};strength_weighted_sum_E";8;TYPE{;<f8;};strength_weighted_gamma_

INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/sources/EXOMOL_test/line_data" in "test_data/test_ld.h5" succeeded
INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/line_data" in "test_data/test_ld.h5" succeeded
DEBUG :: _update_virtual_datasets :: ans_pseudo_continuum_file.py-78 :: s_name='/sources/EXOMOL_test' s_file='.' s_path='/sources/EXOMOL_test/pseudo_continuum'


mol_grp_name='CH4' iso_grp_name='1'
leaf_grp_name='line_set_0000'
leaf_grp_name='line_set_0001'
Performing update...
mol_grp_name='CH4' iso_grp_name='1'
Deleted existing leaf groups...
iso_grp_src_list has been sorted...
len(sorted_iso_grp_src_list)=2
len(sorted_iso_grp_src_list[0])=2
len(sorted_iso_grp_src_list[0][0])=2
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(200.0))
Write virtual dataset
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(250.0))
Write virtual dataset
ADDED LINE DATA...
pc_dfs=PCDataFileSet(t_cont=300.0, mol_spec=(('C', 1), ('H', 4)), iso_name='(12C)(1H)4', s_max=1e-26, contbins=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T300.0.contbins'), continuum=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T300.0.continuum'), stronglines=PosixPath('test_data/EXOMOL_test/12C-1H4__YT34to10_T300.0.stronglines'))
hdr=b'STRUCTURED{;line_strength_sum;0;TYPE{;<f8;};strength_weighted_sum_E";8;TYPE{;<f8;};strength_weighted_gamma_

INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/sources/EXOMOL_test/line_data" in "test_data/test_ld.h5" succeeded
INFO :: _validate_data_group :: ans_line_data_file.py-92 :: Validation for "/line_data" in "test_data/test_ld.h5" succeeded


mol_grp_name='CH4' iso_grp_name='1'
leaf_grp_name='line_set_0000'
leaf_grp_name='line_set_0001'
leaf_grp_name='line_set_0002'
Performing update...
mol_grp_name='CH4' iso_grp_name='1'
Deleted existing leaf groups...
iso_grp_src_list has been sorted...
len(sorted_iso_grp_src_list)=3
len(sorted_iso_grp_src_list[0])=2
len(sorted_iso_grp_src_list[0][0])=2
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(200.0))
Write virtual dataset
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(250.0))
Write virtual dataset
Adding source leaf_grp_src_parameters=(np.float64(1e-26), np.float64(300.0))
Write virtual dataset
ADDED LINE DATA...
